In [1]:
from PIL import Image
import os
import matplotlib.pyplot as plt
import gc
import torch
import torch.nn as nn
from tqdm import tqdm
from torchvision.models import vgg19
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import numpy as np

In [2]:
gc.collect()
torch.cuda.empty_cache()

In [3]:
def scale_down(img, scale):
    img = Image.open(img)
    d_img = img.resize((img.width//scale, img.height//scale), Image.BICUBIC)
    return d_img

In [4]:
def show_img(img1, img2):
    fig, ax = plt.subplots(1, 2, figsize=(10, 5))
    ax[0].imshow(img1)
    ax[0].set_title('low')
    ax[1].imshow(img2)
    ax[1].set_title('high')
    
    plt.show()

In [5]:
scale = 4
train_dir = 'train'
high_dir = os.path.join(train_dir, 'high')
images = os.listdir(high_dir)
l_images = os.listdir(os.path.join(train_dir, 'low'))
for i, image in enumerate(images):
    print(f'{i+1}')
    if image in l_images:
        continue
    u_img = os.path.join(high_dir, image)
    d_img = scale_down(u_img, scale)
    # show_img(d_img, Image.open(u_img))
    d_img.save(os.path.join(train_dir, 'low', image))
    
    torch.cuda.empty_cache()
    gc.collect()
    

1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
277


In [6]:
# Adversarial Loss - Makes images look real
'''
Initially we give noise to generator, it tries to generate an image, it could be a random image. It goes to discriminator with real tag/classification(process says it's real not fake) along with real images. But when it compares generator fake image with real images, it finds out that it's fake. Mostly two type of problem we humans can see:
1. It doesn't look like real image
2. It doesn't look like target image
To make image more realistic we calculate adversarial loss. Adversarial(opposing) relationship between two networks - Generator and Discriminator. Generator tries to forge images to fool discriminator, discriminator tries to spot the forgery. Adversial loss is how often discriminator gets tricked. We tries to minimiae this loss. In this process generator learns to make images appearing more realistic. Now images looks more like real images. It's not guaranteed that it looks like the target image. Now the perceptual loss comes into play.
'''
# Perceptual Loss - Makes them look like the target image
'''
Now generator generates random realistic images which doesn't look like target image. Again discriminator can easily spot the forgery. Now we calcualate perceptual loss between generated image and target image. This mimic how humans perceive images, they doesn't match pixel by pixel but looks feature similarity. Think of an example "Human Seating", now your mind could easily draw an image. We just match that human is there and seating. We don't care how tall he/she is, where he/she is seating, what color cloth he/she is wearing. So perceptual loss captures high level features and tries to minimize the loss. It helps to generate images looking like target image.

Feature extraction training is costly so we use pre-trained models like VGG, ResNet etc.
'''


'\nNow generator generates random realistic images which doesn\'t look like target image. Again discriminator can easily spot the forgery. Now we calcualate perceptual loss between generated image and target image. This mimic how humans perceive images, they doesn\'t match pixel by pixel but looks feature similarity. Think of an example "Human Seating", now your mind could easily draw an image. We just match that human is there and seating. We don\'t care how tall he/she is, where he/she is seating, what color cloth he/she is wearing. So perceptual loss captures high level features and tries to minimize the loss. It helps to generate images looking like target image.\n\nFeature extraction training is costly so we use pre-trained models like VGG, ResNet etc.\n'

In [7]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, use_activation=True, use_batchnorm=True, **kwargs):
        super().__init__()
        self.use_activation = use_activation
        self.cnn = nn.Conv2d(in_channels, out_channels, **kwargs)
        self.bn = nn.BatchNorm2d(out_channels) if use_batchnorm else nn.Identity()
        self.ac = nn.LeakyReLU(0.2, inplace=True)
    
    def forward(self, x):
        cnn = self.cnn(x)
        bn = self.bn(cnn)
        out = self.ac(bn) if self.use_activation else bn
        return out

In [8]:
class UpsampleBlock(nn.Module):
    def __init__(self, in_channels, scale_factor):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, in_channels * scale_factor**2, 2, 1, 1)
        self.ps = nn.PixelShuffle(scale_factor)
        self.ac = nn.PReLU(num_parameters=in_channels)
    
    def forward(self, x):
        return self.ac(self.ps(self.conv(x)))

In [9]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.b1 = ConvBlock(in_channels, in_channels, kernel_size=3, stride=1, padding=1)
        self.b2 = ConvBlock(in_channels, in_channels, kernel_size=3, stride=1, padding=1, use_activation=False)
        
    def forward(self, x):
        b1 = self.b1(x)
        return b1 + self.b2(b1)

In [10]:
class Generator(nn.Module):
    def __init__(self, in_channels=3, num_channels=64, num_blocks=8):
        super().__init__()
        self.initial = ConvBlock(in_channels, num_channels, kernel_size=7, stride=1, padding=4, use_batchnorm=False)
        self.res = nn.Sequential(*[ResidualBlock(num_channels) for _ in range(num_blocks)])
        self.conv = ConvBlock(num_channels, num_channels, kernel_size=3, stride=1, padding=1, use_activation=False)
        self.up = nn.Sequential(UpsampleBlock(num_channels, scale_factor=2))
        self.final = nn.Conv2d(num_channels, in_channels, kernel_size=9, stride=1, padding=1)
    
    def forward(self, x):
        initial = self.initial(x)
        res = self.res(initial)
        conv = self.conv(res) + initial
        up = self.up(conv)
        out = self.final(up)
        return torch.sigmoid(out)

In [11]:
class Discriminator(nn.Module):
    def __init__(self, in_channels=3, features=[64, 64, 128, 128, 256, 256, 512, 512]):
        super().__init__()
        blocks = []
        for i, feature in enumerate(features):
            blocks.append(ConvBlock(in_channels, feature, kernel_size=3, stride = i % 2 + 1, padding=1, use_activation=True, use_batchnorm=i!=0))
            in_channels = feature
        
        self.blocks = nn.Sequential(*blocks)
        self.mlp = nn.Sequential(
            nn.AdaptiveAvgPool2d((8, 8)),
            nn.Flatten(),
            nn.Linear(512*8*8, 1024),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(1024, 1)
        )
    def forward(self, x):
        return self.mlp(self.blocks(x))

In [12]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
lr = 3e-4
epochs = 3
batch_size = 16
num_workers = 0
img_channels = 3

In [13]:
class vggL(nn.Module):
    def __init__(self):
        super().__init__()
        self.vgg = vgg19(weights='DEFAULT').features[:25].eval().to(device)
        self.loss = nn.MSELoss()
    
    def forward(self, first, second):
        vgg_first = self.vgg(first)
        vgg_second = self.vgg(second)
        perceptual_loss = self.loss(vgg_first, vgg_second)
        return perceptual_loss

In [14]:
gen = Generator(in_channels=3).to(device)
disc = Discriminator(in_channels=3).to(device)
opt_gen = torch.optim.Adam(gen.parameters(), lr=lr, betas=(0.9, 0.999))
opt_disc = torch.optim.Adam(disc.parameters(), lr=lr, betas=(0.9, 0.999))
mse = nn.MSELoss()
bce = nn.BCEWithLogitsLoss()
vgg_loss = vggL()

In [15]:
transform_low = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])

transform_high = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((256, 256)),
    transforms.ToTensor()
])

In [16]:
class ImageDataset(Dataset):
    def __init__(self, root_dir):
        super(ImageDataset, self).__init__()
        self.data = []
        self.root_dir = root_dir
        files_low = os.listdir(os.path.join(root_dir, 'low'))
        files_high = os.listdir(os.path.join(root_dir, 'high'))
        self.data = list(zip(files_low, files_high))
        print(self.data)
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, index):
        img_low_file, img_high_file = self.data[index]
        low_res_path = os.path.join(self.root_dir, 'low', img_low_file)
        high_res_path = os.path.join(self.root_dir, 'high', img_high_file)

        low_res = np.array(Image.open(low_res_path))
        high_res = np.array(Image.open(high_res_path))
        
        if len(low_res.shape) != 3:
            low_res = np.stack([low_res] * 3, axis=-1)
        if len(high_res.shape) != 3:
            high_res = np.stack([high_res] * 3, axis=-1)
        low_res = low_res[:, :, :3]
        high_res = high_res[:, :, :3]
        
        low_res = transform_low(low_res)
        high_res = transform_high(high_res)
        
        return low_res, high_res

In [17]:
train_dataset = ImageDataset(root_dir = './train')
train_loader = DataLoader(train_dataset, batch_size=batch_size, num_workers=num_workers)

[('COCO_train2014_000000000009.jpg', 'COCO_train2014_000000000009.jpg'), ('COCO_train2014_000000000030.jpg', 'COCO_train2014_000000000030.jpg'), ('COCO_train2014_000000000049.jpg', 'COCO_train2014_000000000049.jpg'), ('COCO_train2014_000000000077.jpg', 'COCO_train2014_000000000077.jpg'), ('COCO_train2014_000000000089.jpg', 'COCO_train2014_000000000089.jpg'), ('COCO_train2014_000000000092.jpg', 'COCO_train2014_000000000092.jpg'), ('COCO_train2014_000000000109.jpg', 'COCO_train2014_000000000109.jpg'), ('COCO_train2014_000000000110.jpg', 'COCO_train2014_000000000110.jpg'), ('COCO_train2014_000000000127.jpg', 'COCO_train2014_000000000127.jpg'), ('COCO_train2014_000000000142.jpg', 'COCO_train2014_000000000142.jpg'), ('COCO_train2014_000000000144.jpg', 'COCO_train2014_000000000144.jpg'), ('COCO_train2014_000000000194.jpg', 'COCO_train2014_000000000194.jpg'), ('COCO_train2014_000000000250.jpg', 'COCO_train2014_000000000250.jpg'), ('COCO_train2014_000000000308.jpg', 'COCO_train2014_00000000030

In [18]:
def train_fn(train_loader, gen, disc, opt_gen, opt_dis, bce, vggL):
    loop = tqdm(train_loader)
    print(f'Number of batches: {len(loop)}')
    disc_loss = 0
    gen_loss = 0
    
    for i, (low, high) in enumerate(loop):
        low = low.to(device)
        high = high.to(device)

        fake = gen(low)
        disc_real = disc(high)
        disc_fake = disc(fake.detach())
        
        disc_loss_real = bce(disc_real, torch.ones_like(disc_real))
        disc_loss_fake = bce(disc_fake, torch.zeros_like(disc_fake))
        
        disc_loss = disc_loss_fake + disc_loss_real
        
        opt_dis.zero_grad()
        disc_loss.backward()
        opt_dis.step()
        
        disc_fake = disc(fake)
        adversarial_loss = 1e-3 * bce(disc_fake, torch.ones_like(disc_fake))
        vgg_loss = 0.006 * vggL(fake, high)
        gen_loss = vgg_loss + adversarial_loss
        
        opt_gen.zero_grad()
        gen_loss.backward()
        opt_gen.step()
        
    return gen_loss.detach().cpu(), disc_loss.detach().cpu()

In [19]:
d_losses = []
g_losses = []

for epoch in range(epochs):
    gc.collect()
    torch.cuda.empty_cache()
    print(f'Epoch:{epoch+1} / {epochs}')
    gen_loss, disc_loss = train_fn(train_loader, gen, disc, opt_gen, opt_disc, bce, vgg_loss)
    
    d_losses.append(disc_loss)
    g_losses.append(gen_loss)

Epoch:1 / 3


  0%|          | 0/276 [00:00<?, ?it/s]

Number of batches: 276


100%|██████████| 276/276 [3:58:42<00:00, 51.89s/it]  


Epoch:2 / 3


...

100%|██████████| 276/276 [4:04:00<00:00, 53.04s/it]  


Epoch:3 / 3


  0%|          | 0/276 [00:00<?, ?it/s]

Number of batches: 276


100%|██████████| 276/276 [3:32:45<00:00, 46.25s/it]  


In [20]:
torch.save(gen.state_dict(), 'checkpoint1_gen.pth')